<a id='9c-12'></a>

## 9c-12. 🧩 Pattern 12: Unpacking & Starred Expressions — LC 15, 56

---

```
PROBLEM:  Destructure sequences and dicts into named pieces without index gymnastics.

APPROACH: Use tuple unpacking, *rest catching, and **kwargs spreading to pull
          values apart cleanly. Works in assignments, function calls, and loops.

SLOW MOTION TRACE:
  # Basic tuple unpack
  a, b, c = (1, 2, 3)
  step 1   a=1, b=2, c=3   (one assignment, three names)

  # Star — catch the "rest"
  first, *middle, last = [1, 2, 3, 4, 5]
  step 1   first=1, middle=[2,3,4], last=5

  # Swap without temp variable
  a, b = 3, 7
  a, b = b, a            # Python packs right side as tuple first, then unpacks
  step 1   a=7, b=3

  # Spread into function call
  def add(x, y, z): return x+y+z
  args = [1, 2, 3]
  add(*args)             -> 6       (* spreads list into positional args)

  # Merge dicts (Python 3.9+ | operator, older: {**d1, **d2})
  d1 = {'a': 1}; d2 = {'b': 2}
  merged = {**d1, **d2}  -> {'a':1, 'b':2}

KEY INSIGHT: Unpacking eliminates magic indices like row[0], row[1] — the name
             itself documents what the value means.

TIME / SPACE: O(n) for starred unpack (builds a list), O(1) for fixed unpack.
```


In [ ]:
from typing import List, Tuple


# ── Basic tuple unpacking ─────────────────────────────────────────────────────
point = (3, 7)
x, y = point                       # names tell you exactly what each value is
print(f"x={x}, y={y}")             # x=3, y=7

# nested unpack
((x1, y1), (x2, y2)) = ((0, 0), (4, 3))
print(f"p1=({x1},{y1})  p2=({x2},{y2})")   # p1=(0,0) p2=(4,3)


# ── Starred * — catch the rest ───────────────────────────────────────────────
first, *middle, last = [10, 20, 30, 40, 50]
print(f"first={first}  middle={middle}  last={last}")
# first=10  middle=[20,30,40]  last=50

# head / tail split — classic functional pattern
head, *tail = [1, 2, 3, 4]
print(f"head={head}  tail={tail}")          # head=1  tail=[2,3,4]


# ── Swap without temp ────────────────────────────────────────────────────────
a, b = 3, 7
a, b = b, a            # Python builds (b,a) tuple on the right, then unpacks left
print(f"after swap: a={a}, b={b}")          # a=7, b=3


# ── Unpacking in for loops ───────────────────────────────────────────────────
pairs = [(1, 'one'), (2, 'two'), (3, 'three')]
for num, word in pairs:            # unpack each tuple as you iterate
    print(f"  {num} -> {word}")


# ── Spread * into function calls ─────────────────────────────────────────────
def add3(x, y, z): return x + y + z

args = [1, 2, 3]
print(add3(*args))      # 6  — * spreads list into positional params

def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

kwargs = {"name": "Sean", "greeting": "Hey"}
print(greet(**kwargs))  # Hey, Sean!  — ** spreads dict into keyword params


# ── Merge dicts ──────────────────────────────────────────────────────────────
defaults = {"timeout": 30, "retries": 3}
overrides = {"retries": 5, "verbose": True}
config = {**defaults, **overrides}   # overrides wins on key collision
print(config)           # {'timeout': 30, 'retries': 5, 'verbose': True}


# ── LC 56 — Merge Intervals (unpack interval edges cleanly) ──────────────────
def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    """
    LC 56 — Merge Intervals.
    Approach: sort by start, merge overlapping neighbors.
    Args:
        intervals (List[List[int]]): list of [start, end] intervals.
    Returns:
        List[List[int]]: merged non-overlapping intervals.
    Time:  O(n log n) — dominated by sort
    Space: O(n) — result list
    """
    intervals.sort(key=lambda iv: iv[0])    # sort by start
    merged = [intervals[0]]                  # seed with first interval

    for start, end in intervals[1:]:         # unpack — no iv[0], iv[1] magic
        prev_start, prev_end = merged[-1]    # unpack last merged interval
        if start <= prev_end:                # overlap: current start inside previous
            merged[-1] = [prev_start, max(prev_end, end)]   # extend
        else:
            merged.append([start, end])      # gap: new separate interval

    return merged

# Slow motion on [[1,3],[2,6],[8,10],[15,18]]:
# seed: [[1,3]]
# [2,6]: 2<=3 overlap -> extend to [1,6]  -> [[1,6]]
# [8,10]: 8>6 gap     -> append           -> [[1,6],[8,10]]
# [15,18]: 15>10 gap  -> append           -> [[1,6],[8,10],[15,18]]

def test_harness_merge(fn):
    tests = [
        ([[1,3],[2,6],[8,10],[15,18]], [[1,6],[8,10],[15,18]]),
        ([[1,4],[4,5]], [[1,5]]),
        ([[1,4],[2,3]], [[1,4]]),
        ([[1,2]], [[1,2]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_merge(merge_intervals)
print("merge_intervals defined.")

# Simplicity and clarity is Gold
